In [1]:
import pandas as pd
import numpy as np

In [2]:
wellness_master = pd.read_csv(
    "../Data/Processed/wellness/wellness_master.csv"
)

training_master = pd.read_csv(
    "../Data/Processed/training_load/training_load_master.csv"
)

game_performance = pd.read_csv(
    "../Data/subjective/game-performance/game-performance.csv"
)

In [3]:
print("Wellness:", wellness_master.shape)
print("Training:", training_master.shape)
print("Performance:", game_performance.shape)

Wellness: (36550, 8)
Training: (36550, 9)
Performance: (248, 5)


In [4]:
training_master = training_master.rename(
    columns={
        "Date": "date",
        "player_id": "player_name"
    }
)

game_performance = game_performance.rename(
    columns={
        "timestamp": "date"
    }
)

In [5]:
#convert dates 
training_master["date"] = pd.to_datetime(
    training_master["date"],
    format="%d.%m.%Y"
)

game_performance["date"] = pd.to_datetime(
    game_performance["date"],
    format="%d.%m.%Y"
)

wellness_master["date"] = pd.to_datetime(
    wellness_master["date"],
    format="%d.%m.%Y"
)

In [6]:
print(training_master["date"].dtype)
print(wellness_master["date"].dtype)
print(game_performance["date"].dtype)

datetime64[us]
datetime64[us]
datetime64[us]


In [7]:
wellness_master.columns

Index(['date', 'Player Name', 'fatigue', 'mood', 'readiness', 'sleep_duration',
       'soreness', 'stress'],
      dtype='str')

In [8]:
#Merging wellness and training data
master = training_master.merge(
    wellness_master,
    left_on=["date", "player_name"],
    right_on=["date", "Player Name"],
    how="left"
)

master.shape

(36550, 16)

In [9]:
master.head()

,date,player_name,acwr,atl,ctl28,ctl42,daily_load,monotony,strain,Player Name,fatigue,mood,readiness,sleep_duration,soreness,stress
0,2020-01-01,TeamA-0362cdd5-7a63-480a-a46a-62a99fb1692f,0.0,0.0,0.0,0.0,0.0,0.0,0.0,TeamA-0362cdd5-7a63-480a-a46a-62a99fb1692f,NaN,NaN,NaN,NaN,NaN,NaN
1,2020-01-01,TeamA-2d44f941-2f24-4fc2-afa8-611a091f2e93,0.0,0.0,0.0,0.0,0.0,0.0,0.0,TeamA-2d44f941-2f24-4fc2-afa8-611a091f2e93,NaN,NaN,NaN,NaN,NaN,NaN
2,2020-01-01,TeamA-32fed4b3-d7fc-482d-ba21-c46c58f015b5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,TeamA-32fed4b3-d7fc-482d-ba21-c46c58f015b5,NaN,NaN,NaN,NaN,NaN,NaN
3,2020-01-01,TeamA-358603ef-b3a3-46b5-b80a-ad64e06b6592,0.0,0.0,0.0,0.0,0.0,0.0,0.0,TeamA-358603ef-b3a3-46b5-b80a-ad64e06b6592,NaN,NaN,NaN,NaN,NaN,NaN
4,2020-01-01,TeamA-3e5f6e2b-46b7-4890-84a9-3bbb2649af5a,0.0,0.0,0.0,0.0,0.0,0.0,0.0,TeamA-3e5f6e2b-46b7-4890-84a9-3bbb2649af5a,NaN,NaN,NaN,NaN,NaN,NaN


In [10]:
#chceking for missing values
master[
    [
        "fatigue",
        "mood",
        "readiness",
        "sleep_duration",
        "soreness",
        "stress"
    ]
].isnull().sum()

fatigue           19558
mood              19551
readiness         19553
sleep_duration    19569
soreness          19551
stress            19553
dtype: int64

In [12]:
master[
    master["fatigue"].notna()
][
    ["date", "player_name", "fatigue"]
].head(10)

,date,player_name,fatigue
51,2021-01-01,TeamA-2d44f941-2f24-4fc2-afa8-611a091f2e93,2.0
59,2021-01-01,TeamA-5cd7a61b-88b2-46d2-94f8-5a0d4f682d93,3.0
61,2021-01-01,TeamA-705923a2-378f-4f8e-8a29-464837b88cdc,4.0
62,2021-01-01,TeamA-74afe68c-f348-414c-9754-6d6f9df12587,2.0
69,2021-01-01,TeamA-b58af410-da77-479e-b93c-e03617b9f36d,2.0
73,2021-01-01,TeamA-d7299614-fa73-4f69-b5e9-f913e3154ff6,2.0
75,2021-01-01,TeamA-ecdbd8ec-61a7-4131-97ec-3af76f621f65,3.0
98,2021-01-01,TeamB-e4321aef-964a-4fdd-88bd-0bd548b0e93b,2.0
129,2020-02-01,TeamB-2f23d7d5-2326-49ce-b9c8-5a6303f785c5,4.0
130,2020-02-01,TeamB-4405bb1f-56f7-48ba-bfa8-e795e4006952,4.0


In [13]:
master = master.drop(columns=["Player Name"])

master.shape

(36550, 15)

In [14]:
master = master.merge(
    game_performance,
    on=["player_name", "date"],
    how="left"
)

master.shape

(36551, 18)

In [15]:
master[
    [
        "team_performance",
        "offensive_performance",
        "defensive_performance"
    ]
].notna().sum()

team_performance         248
offensive_performance    248
defensive_performance    248
dtype: int64

In [16]:
game_performance.duplicated(
    subset=["player_name", "date"]
).sum()

np.int64(1)

In [17]:
game_performance[
    game_performance.duplicated(
        subset=["player_name", "date"],
        keep=False
    )
].sort_values(["player_name", "date"])

,player_name,team_performance,offensive_performance,defensive_performance,date
236,TeamB-6d568bee-175f-4dcb-9d3a-0f3e8f35de07,7,7,7,2021-09-11
237,TeamB-6d568bee-175f-4dcb-9d3a-0f3e8f35de07,7,7,7,2021-09-11


# Get All GPS FILES 

In [18]:
from pathlib import Path

gps_files = list(
    Path("../Data/2020").rglob("*.parquet")
)

print("Total GPS Files:", len(gps_files))

Total GPS Files: 2325


In [20]:
#Created Emepty list to store session summaries
all_sessions = []

In [21]:
for file_path in gps_files:

    df = pd.read_parquet(file_path)

    session_date = file_path.parent.name

    session_summary = {
        "player_name": df["player_name"].iloc[0],

        "date": pd.to_datetime(session_date),

        "avg_speed": df["speed"].mean(),
        "max_speed": df["speed"].max(),

        "avg_heart_rate": df["heart_rate"].mean(),
        "max_heart_rate": df["heart_rate"].max(),

        "avg_accel_x": df["accl_x"].mean(),
        "avg_accel_y": df["accl_y"].mean(),
        "avg_accel_z": df["accl_z"].mean(),

        "rows": len(df)
    }

    all_sessions.append(session_summary)

In [22]:
gps_master = pd.DataFrame(all_sessions)

gps_master.head()

,player_name,date,avg_speed,max_speed,avg_heart_rate,max_heart_rate,avg_accel_x,avg_accel_y,avg_accel_z,rows
0,TeamA-1846d424-c17c-6279-23c6-612f48268673,2020-06-01,0.988182,6.575005,145.431809,196,-0.024538,0.879062,0.530340,680300
1,TeamA-23a7711a-8133-2876-37eb-dcd9e87a1613,2020-06-01,1.009975,6.955561,31.040843,181,0.015004,0.886508,0.516305,698290
2,TeamA-2d44f941-2f24-4fc2-afa8-611a091f2e93,2020-06-01,0.939565,6.455561,147.744218,198,-0.060387,0.848780,0.680267,696160
3,TeamA-32fed4b3-d7fc-482d-ba21-c46c58f015b5,2020-06-01,1.108051,7.100006,154.653994,190,0.031907,0.920755,0.514396,656390
4,TeamA-3e5f6e2b-46b7-4890-84a9-3bbb2649af5a,2020-06-01,1.043769,7.422228,149.098588,195,-0.058876,0.978022,0.469338,692480


In [23]:
gps_master.shape

(2325, 10)